# 06. Backtracking für große Fälle: geschickt verzweigen, vorausschauend prüfen, Sackgassen merken

In [Notebook 02 von learn_bwinf1](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/02_backtracking.ipynb) hast du Backtracking gelernt: eine Lösung Schritt für Schritt aufbauen und zurücknehmen, wenn es nicht weitergeht. Dort war der Suchbaum klein. Bei den Aufgaben der 2. Runde ist er riesig, und **dasselbe Verfahren** braucht entweder Sekunden oder Jahrtausende, je nachdem, **in welcher Reihenfolge** du die Entscheidungen triffst und **wie früh** du Sackgassen erkennst. Dazu kommen [Schranken und Symmetrien](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/03_schranken_symmetrien.ipynb) und das [logische Schließen](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/04_logisch_schliessen.ipynb) aus dem ersten Lehrgang, die du hier vertiefst, statt sie zu wiederholen.

In diesem Notebook lernst du:

- die **Knoten der Suche zu zählen** und jede Verbesserung mit einer **Tabelle** (Verfahren × Beispiel → besuchte Knoten) zu messen, statt zu raten,
- **wo man verzweigt**: erst das Feld mit den **wenigsten Möglichkeiten** (Funktion als Argument der Suche),
- **vorausschauend zu prüfen**: erzwungene Schritte sofort ausführen und Widersprüche früh erkennen,
- **Teilzustände zu merken** (Sackgassen, gleiche Teile zusammenzufassen) und **Symmetrien** wegzuschneiden,
- den Unterschied zwischen **einer Lösung finden**, **alle zählen** und **beweisen, dass es keine gibt**,
- die Ergebnisse mit einem **Schiedsrichter** zu prüfen ([Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb)) und am Ende mit einem **Solver** (MiniZinc) zu vergleichen.

Die lokale Suche aus [Notebook 04](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/04_lokale_suche.ipynb) findet gute Lösungen, aber sie **beweist** nie, dass es keine bessere oder gar keine Lösung gibt. Hier geht es um die Fälle, in denen du **Sicherheit** brauchst, und dafür muss die vollständige Suche **groß genug** für die Eingabe werden.

Die neutralen Beispiele sind **Sudoku** und das **Füllen eines Rechtecks mit Pentominos**. Danach wendest du die Ideen in zwei Ecken auf Aufgaben der 2. Runde an: **Konfetti** (43. BwInf) und, für Schnelle, **Komplementär** (31. BwInf). Wenn du eine Aufgabe nicht schaffst, kopierst du die Lösung aus dem aufklappbaren Teil und führst sie aus, damit die späteren Zellen laufen.

## 1. Sudoku: Der Suchbaum und seine Knoten

Ein Sudoku hat 81 Felder in einem 9 × 9-Raster. In jeder Zeile, jeder Spalte und jedem 3 × 3-Block muss jede Ziffer 1 bis 9 genau einmal stehen. Einige Felder sind vorgegeben, die anderen sollst du füllen.

Wir speichern das Raster als **Liste von 81 Zahlen** (0 = leer). Das Feld in Zeile `zeile` und Spalte `spalte` hat die Nummer `9 * zeile + spalte`. `NACHBARN[feld]` ist die Liste aller Felder, die mit `feld` in einer Zeile, Spalte oder einem Block liegen (20 Stück). `pruefe_sudoku` ist der **Schiedsrichter** wie in [Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb): Er prüft jede Lösung, die wir gleich finden. Die vier Beispiele heißen `leicht`, `mittel`, `schwer` und `brutal` (der Name sagt, wie gut die Suche mit ihnen zurechtkommt).

In [ ]:
def nachbarn_von(feld):
    """Alle Felder, die in derselben Zeile, Spalte oder demselben 3x3-Block liegen (ohne das Feld selbst)."""
    zeile = feld // 9
    spalte = feld % 9
    menge = set()
    for k in range(9):
        menge.add(zeile * 9 + k)
        menge.add(k * 9 + spalte)
    z0 = zeile // 3 * 3
    s0 = spalte // 3 * 3
    for a in range(3):
        for b in range(3):
            menge.add((z0 + a) * 9 + s0 + b)
    menge.discard(feld)
    return sorted(menge)


NACHBARN = []
for feld in range(81):
    NACHBARN.append(nachbarn_von(feld))

TEXTE = {
    'leicht': '530070000600195000098000060800060003400803001700020006060000280000419005000080079',
    'mittel': '000000907000420180000705026100904000050000040000507009920108000034059000507000000',
    'schwer': '400000805030000000000700000020000060000080400000010000000603070500200000104000000',
    'brutal': '000000000000003085001020000000507000004000100090000000500000073002010000000040009',
}


def lies_sudoku(text):
    """Eine Zeichenkette aus 81 Ziffern (0 = leer) wird zur Liste von 81 Zahlen. Feld Nummer 9 * Zeile + Spalte."""
    raster = []
    for zeichen in text:
        raster.append(int(zeichen))
    return raster


def zeige_sudoku(raster):
    for zeile in range(9):
        teile = []
        for spalte in range(9):
            ziffer = raster[zeile * 9 + spalte]
            teile.append(str(ziffer) if ziffer > 0 else '.')
        print(' '.join(teile[0:3]), '|', ' '.join(teile[3:6]), '|', ' '.join(teile[6:9]))
        if zeile % 3 == 2 and zeile < 8:
            print('------+-------+------')


def pruefe_sudoku(raster, vorgabe):
    """Schiedsrichter: 'ok' oder die Beschreibung des ersten Fehlers."""
    for feld in range(81):
        if raster[feld] < 1 or raster[feld] > 9:
            return f'Feld {feld} ist nicht mit 1 bis 9 gefüllt'
        if vorgabe[feld] != 0 and raster[feld] != vorgabe[feld]:
            return f'Feld {feld} ändert die Vorgabe {vorgabe[feld]}'
    for feld in range(81):
        for nachbar in NACHBARN[feld]:
            if raster[feld] == raster[nachbar]:
                return f'Die Felder {feld} und {nachbar} haben beide die {raster[feld]}'
    return 'ok'


leicht = lies_sudoku(TEXTE['leicht'])
zeige_sudoku(leicht)
print('leere Felder:', leicht.count(0))

#### Aufgabe 1.1

Schreibe `kandidaten(raster, feld)`: Sie liefert die Liste der Ziffern von 1 bis 9, die noch **nicht** bei einem Nachbarn des Feldes stehen. Mit `leicht` (siehe oben) ist das Feld 2 die dritte Zelle der ersten Zeile, Feld 4 steht in der Mitte der ersten Zeile.

**Erwartete Ausgabe:**
```
[1, 2, 4]
[4, 7]
```

In [ ]:
def kandidaten(raster, feld):
    """Die Ziffern, die man auf das Feld schreiben darf (kein Nachbar hat sie schon)."""
    # TODO
    pass


print(kandidaten(leicht, 2))
print(kandidaten(leicht, 4))

<details>
<summary>Tipp anzeigen</summary>

Sammle die Ziffern der Nachbarn in einer Menge (`set`). Dann gehst du die Ziffern 1 bis 9 durch und nimmst die, die nicht in der Menge sind.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def kandidaten(raster, feld):
    """Die Ziffern, die man auf das Feld schreiben darf (kein Nachbar hat sie schon)."""
    belegt = set()
    for nachbar in NACHBARN[feld]:
        belegt.add(raster[nachbar])
    moeglich = []
    for ziffer in range(1, 10):
        if ziffer not in belegt:
            moeglich.append(ziffer)
    return moeglich


print(kandidaten(leicht, 2))
print(kandidaten(leicht, 4))
```

</details>

#### Frage 1.2

`leicht` hat 51 leere Felder.

1. Wie viele Belegungen gibt es, wenn man **jedes** leere Feld mit **jeder** Ziffer von 1 bis 9 füllt, ohne auf die Regeln zu achten? Schätze die Größenordnung.
2. Warum probiert Backtracking trotzdem nur einen winzigen Bruchteil davon aus?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Es sind 9^51. Das ist eine Zahl mit 49 Stellen (`len(str(9 ** 51))` rechnet das aus), nicht durchzuprobieren.
2. Backtracking setzt eine Ziffer nur, wenn sie **jetzt** zu keinem Nachbarn passt (`kandidaten`), und schneidet damit **den ganzen Teilbaum** darunter ab: Eine falsche Ziffer in der ersten Zeile zerstört alle 9^50 Fortsetzungen auf einmal, sobald sie sich widersprechen. Wie groß der Baum **tatsächlich** wird, hängt davon ab, wie früh man Widersprüche bemerkt. Genau das untersuchen wir jetzt.

</details>

### Die Suche und ihre Knoten

Die Suche ist die aus [learn_bwinf1, Notebook 02](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/02_backtracking.ipynb): Wähle ein leeres Feld, probiere der Reihe nach seine Kandidaten, gehe rekursiv weiter und nimm die Ziffer zurück, wenn es nicht klappt. Neu sind zwei Dinge:

- Ein **Knoten** ist ein **Aufruf** von `suche`. Wir zählen sie in einem Wörterbuch `zaehler`. Die Zahl der Knoten ist **unser Maß für die Größe des Suchbaums**: unabhängig vom Computer, im Browser und hier gleich.
- Die Auswahl des Feldes übernimmt eine **Funktion** `wahl`, die wir der Suche als **Argument** übergeben. Eine Funktion ist in Python ein Wert wie eine Zahl: `wahl_erstes` (ohne Klammern) ist die Funktion selbst, `wahl_erstes(raster)` ein Aufruf. In `suche` steht dann `wahl(raster)`, und wir können die Auswahlregel austauschen, **ohne die Suche zu ändern**.

Weil die Suche manchmal zu lange dauern würde, hat der Zähler eine **Grenze** (hier 50 000 Knoten). Wird sie erreicht, ist die Suche `abgebrochen`. Das heißt **nicht**, dass es keine Lösung gibt, sondern nur, dass wir aufgegeben haben.

#### Aufgabe 1.3

Schreibe `suche` fertig. Die Zeilen zum Zählen und zum Abbrechen stehen schon im Gerüst. Setze das **erste leere Feld** (`wahl_erstes`) und probiere alle Kandidaten. Die Zelle gibt aus, was der Schiedsrichter sagt und wie viele Knoten es waren.

**Erwartete Ausgabe:**
```
ok , 4209 Knoten
```

In [ ]:
def neuer_zaehler(grenze=50000):
    return {'knoten': 0, 'grenze': grenze, 'abgebrochen': False}


def wahl_erstes(raster):
    """Das erste leere Feld (None, wenn es kein leeres Feld mehr gibt)."""
    for feld in range(81):
        if raster[feld] == 0:
            return feld
    return None


def suche(raster, wahl, zaehler):
    """Backtracking. wahl ist eine Funktion: wahl(raster) liefert das nächste Feld. Gibt die Lösung zurück oder None."""
    zaehler['knoten'] += 1
    if zaehler['knoten'] > zaehler['grenze']:
        zaehler['abgebrochen'] = True
        return None
    # TODO
    return None


vorgabe = lies_sudoku(TEXTE['leicht'])
zaehler = neuer_zaehler()
ergebnis = suche(vorgabe[:], wahl_erstes, zaehler)
if ergebnis is None:
    print('keine Lösung gefunden,', zaehler['knoten'], 'Knoten')
else:
    print(pruefe_sudoku(ergebnis, vorgabe), ',', zaehler['knoten'], 'Knoten')

<details>
<summary>Tipp anzeigen</summary>

Rufe `wahl(raster)` auf. Ist das Ergebnis `None`, ist alles gefüllt: gib eine **Kopie** des Rasters zurück (`raster[:]`). Sonst setze nacheinander jede Ziffer aus `kandidaten`, rufe `suche` rekursiv auf und setze das Feld danach auf 0 zurück.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def neuer_zaehler(grenze=50000):
    return {'knoten': 0, 'grenze': grenze, 'abgebrochen': False}


def wahl_erstes(raster):
    """Das erste leere Feld (None, wenn es kein leeres Feld mehr gibt)."""
    for feld in range(81):
        if raster[feld] == 0:
            return feld
    return None


def suche(raster, wahl, zaehler):
    """Backtracking. wahl ist eine Funktion: wahl(raster) liefert das nächste Feld. Gibt die Lösung zurück oder None."""
    zaehler['knoten'] += 1
    if zaehler['knoten'] > zaehler['grenze']:
        zaehler['abgebrochen'] = True
        return None
    feld = wahl(raster)
    if feld is None:
        return raster[:]                          # alles gefüllt: eine Kopie ist die Lösung
    for ziffer in kandidaten(raster, feld):
        raster[feld] = ziffer
        ergebnis = suche(raster, wahl, zaehler)
        raster[feld] = 0                          # zurücknehmen
        if ergebnis is not None or zaehler['abgebrochen']:
            return ergebnis
    return None


vorgabe = lies_sudoku(TEXTE['leicht'])
zaehler = neuer_zaehler()
ergebnis = suche(vorgabe[:], wahl_erstes, zaehler)
if ergebnis is None:
    print('keine Lösung gefunden,', zaehler['knoten'], 'Knoten')
else:
    print(pruefe_sudoku(ergebnis, vorgabe), ',', zaehler['knoten'], 'Knoten')
```

</details>

Schon bei `leicht` sind es mehrere Tausend Knoten, obwohl die Aufgabe leicht heißt: Die Suche geht stur von links oben nach rechts unten und stößt oft erst viel später auf einen Widerspruch.

### Wo verzweigen: das Feld mit den wenigsten Möglichkeiten

Wenn ein Feld **nur einen** Kandidaten hat, gibt es dort keine Wahl, der Baum verzweigt nicht. Hat ein Feld **keinen**, ist der Zweig sofort tot. Die Idee: **Verzweige dort, wo es am wenigsten Möglichkeiten gibt** (englisch *most constrained first* oder *fail first*). Dann sind die Verzweigungen im Baum **schmal**, und Widersprüche tauchen **weit oben** auf, wo ein abgeschnittener Teilbaum noch riesig ist.

#### Aufgabe 1.4

Schreibe `wahl_wenigste(raster)`: Sie liefert das leere Feld mit den **wenigsten Kandidaten** (bei Gleichstand das erste) oder `None`, wenn es kein leeres Feld mehr gibt. Die Suche bleibt unverändert, nur die Auswahlfunktion wird ausgetauscht.

**Erwartete Ausgabe:**
```
ok , 52 Knoten
```

In [ ]:
def wahl_wenigste(raster):
    """Das leere Feld mit den wenigsten Kandidaten (None, wenn alles gefüllt ist)."""
    # TODO
    return None


vorgabe = lies_sudoku(TEXTE['leicht'])
zaehler = neuer_zaehler()
ergebnis = suche(vorgabe[:], wahl_wenigste, zaehler)
if ergebnis is None:
    print('keine Lösung gefunden,', zaehler['knoten'], 'Knoten')
else:
    print(pruefe_sudoku(ergebnis, vorgabe), ',', zaehler['knoten'], 'Knoten')

<details>
<summary>Tipp anzeigen</summary>

Merke dir das beste Feld und seine Zahl der Kandidaten (am Anfang 10). Gehe alle leeren Felder durch und ersetze das beste Feld nur, wenn ein Feld **echt weniger** Kandidaten hat.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def wahl_wenigste(raster):
    """Das leere Feld mit den wenigsten Kandidaten (None, wenn alles gefüllt ist)."""
    bestes = None
    wenigste = 10
    for feld in range(81):
        if raster[feld] == 0:
            anzahl = len(kandidaten(raster, feld))
            if anzahl < wenigste:
                bestes = feld
                wenigste = anzahl
    return bestes


vorgabe = lies_sudoku(TEXTE['leicht'])
zaehler = neuer_zaehler()
ergebnis = suche(vorgabe[:], wahl_wenigste, zaehler)
if ergebnis is None:
    print('keine Lösung gefunden,', zaehler['knoten'], 'Knoten')
else:
    print(pruefe_sudoku(ergebnis, vorgabe), ',', zaehler['knoten'], 'Knoten')
```

</details>

### Die Tabelle: Verfahren × Beispiel → besuchte Knoten

Ab jetzt messen wir jede Verbesserung so. Die Funktion `tabelle` führt die Suche für jedes Verfahren auf allen vier Beispielen aus, prüft jedes Ergebnis mit dem Schiedsrichter und schreibt die Zahl der Knoten in die Tabelle. Wo die Grenze von 50 000 Knoten erreicht wird, steht `> 50 000`.

In [ ]:
def knoten_text(name, wahl, voraus=False):
    """Zahl der Knoten als Text; '> 50 000', wenn die Grenze erreicht wurde."""
    zaehler = neuer_zaehler()
    raster = lies_sudoku(TEXTE[name])
    if voraus:
        ergebnis = suche_voraus(raster, wahl, zaehler)
    else:
        ergebnis = suche(raster, wahl, zaehler)
    if zaehler['abgebrochen']:
        return '> 50 000'
    assert pruefe_sudoku(ergebnis, lies_sudoku(TEXTE[name])) == 'ok'
    return f"{zaehler['knoten']:,}".replace(',', ' ')


def tabelle(verfahren):
    """verfahren: Liste von (Name, wahl, voraus)."""
    namen = ['leicht', 'mittel', 'schwer', 'brutal']
    print(f'{"Verfahren":<30}' + ''.join(f'{n:>10}' for n in namen))
    for titel, wahl, voraus in verfahren:
        print(f'{titel:<30}' + ''.join(f'{knoten_text(n, wahl, voraus):>10}' for n in namen))

In [ ]:
tabelle([('erstes leeres Feld', wahl_erstes, False),
         ('wenigste Kandidaten', wahl_wenigste, False)])

Verfahren                         leicht    mittel    schwer    brutal
erstes leeres Feld                 4 209    19 023  > 50 000  > 50 000
wenigste Kandidaten                   52       117       482    45 268


**Lies die Tabelle:** Mit dem Feld mit den wenigsten Möglichkeiten sinkt die Zahl der Knoten bei `leicht` von 4 209 auf 52, bei `mittel` von 19 023 auf 117 und bei `schwer` von `> 50 000` auf 482. Bei `brutal` bricht das erste Verfahren ab (`> 50 000`), das zweite findet die Lösung mit 45 268 Knoten, gerade noch unter der Grenze. Das Rechnen pro Knoten ist bei `wahl_wenigste` teurer (sie prüft alle leeren Felder), aber ein Knoten ist hier so viel mehr wert, dass sich das lohnt.

#### Frage 1.5

1. Warum hilft „wenigste Kandidaten zuerst“? Erkläre mit den beiden Sonderfällen **0 Kandidaten** und **1 Kandidat**.
2. Warum ist `wahl_wenigste` pro Knoten **teurer** als `wahl_erstes`? Wie viele Aufrufe von `kandidaten` macht sie höchstens pro Knoten?
3. Für `schwer` und `brutal` braucht „erstes freies Feld“ mehr als 50 000 Knoten, die Suche bricht ab. Heißt das, dass die beiden keine Lösung haben?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Bei **0 Kandidaten** weiß man sofort, dass dieser Zweig tot ist, und kann ihn **abschneiden**: Dieses Feld findet `wahl_wenigste` zuerst. Bei **1 Kandidat** gibt es nichts zu probieren: Die Verzweigung hat nur einen Ast, der Baum wird nicht breiter. Wer stur von links oben füllt, merkt einen Widerspruch (ein Feld ohne Kandidaten) oft erst viele Ebenen später und probiert vorher unzählige sinnlose Ziffern in den Feldern dazwischen durch. **Je früher ein Widerspruch auftaucht, desto größer der abgeschnittene Teilbaum.**
2. `wahl_erstes` sucht nur das erste leere Feld (höchstens 81 Schritte, meist wenige). `wahl_wenigste` ruft für **jedes** leere Feld `kandidaten` auf, bis zu 81 Aufrufe mit je 20 Nachbarn, also etwa 81 · 20 Schritte pro Knoten. Man tauscht also **teurere Knoten gegen weniger Knoten**. Wichtig ist, was **insgesamt** weniger kostet.
3. **Nein.** `abgebrochen` heißt nur, dass die Grenze erreicht wurde. Nur eine **vollständige** Suche, die mit „keine Lösung“ **endet**, beweist die Unlösbarkeit. Beide haben sogar eine Lösung, wie wir gleich sehen.

</details>

### Vorausschauend prüfen: erzwungene Felder sofort setzen

Noch besser: Hat ein Feld nur **einen** Kandidaten, dann **setze ihn sofort**. Das verändert die Kandidaten der Nachbarn, vielleicht gibt es dann das nächste erzwungene Feld, und so weiter. Hat dabei ein leeres Feld **keine** Möglichkeit mehr, ist der Zweig tot, **ohne dass wir eine einzige Ziffer raten** mussten. Das ist das **logische Schließen** aus [learn_bwinf1, Notebook 04](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/04_logisch_schliessen.ipynb), jetzt in der Suche. Man nennt es **Propagation**: Eine Entscheidung wird über die Regeln „weitergereicht“, bis nichts mehr folgt.

#### Aufgabe 1.6

Schreibe `einzelne_setzen(raster)`. Sie setzt wiederholt jedes leere Feld mit **genau einem** Kandidaten, bis sich nichts mehr ändert. Hat ein leeres Feld **keinen** Kandidaten, gibt sie `False` zurück (Widerspruch), sonst `True`. Die Zelle probiert sie mit `leicht` aus: Wie viele Felder bleiben leer?

**Erwartete Ausgabe:**
```
True 0 leere Felder
```

In [ ]:
def einzelne_setzen(raster):
    """Setzt alle Felder, die nur noch eine Möglichkeit haben, und das immer wieder, bis nichts mehr geht.
    Gibt False zurück, wenn ein leeres Feld gar keine Möglichkeit mehr hat (Widerspruch), sonst True."""
    # TODO
    return True


probe = lies_sudoku(TEXTE['leicht'])
print(einzelne_setzen(probe), probe.count(0), 'leere Felder')

<details>
<summary>Tipp anzeigen</summary>

Eine `while`-Schleife mit einer Variablen `geaendert`: Setze sie am Anfang jeder Runde auf `False` und bei jedem gesetzten Feld auf `True`. Du darfst das Raster direkt ändern, denn der Aufrufer übergibt eine Kopie.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def einzelne_setzen(raster):
    """Setzt alle Felder, die nur noch eine Möglichkeit haben, und das immer wieder, bis nichts mehr geht.
    Gibt False zurück, wenn ein leeres Feld gar keine Möglichkeit mehr hat (Widerspruch), sonst True."""
    geaendert = True
    while geaendert:
        geaendert = False
        for feld in range(81):
            if raster[feld] == 0:
                moeglich = kandidaten(raster, feld)
                if len(moeglich) == 0:
                    return False
                if len(moeglich) == 1:
                    raster[feld] = moeglich[0]
                    geaendert = True
    return True


probe = lies_sudoku(TEXTE['leicht'])
print(einzelne_setzen(probe), probe.count(0), 'leere Felder')
```

</details>

In [ ]:
def suche_voraus(raster, wahl, zaehler):
    """Wie suche, aber nach jedem Setzen werden zuerst alle erzwungenen Felder gesetzt."""
    zaehler['knoten'] += 1
    if zaehler['knoten'] > zaehler['grenze']:
        zaehler['abgebrochen'] = True
        return None
    raster = raster[:]                            # eigene Kopie: Wir müssen nichts zurücknehmen
    if not einzelne_setzen(raster):
        return None
    feld = wahl(raster)
    if feld is None:
        return raster
    for ziffer in kandidaten(raster, feld):
        raster[feld] = ziffer
        ergebnis = suche_voraus(raster, wahl, zaehler)
        if ergebnis is not None or zaehler['abgebrochen']:
            return ergebnis
    return None

`suche_voraus` ist `suche` mit zwei Änderungen: Am Anfang jedes Knotens werden die erzwungenen Felder gesetzt (und bei einem Widerspruch kehrt die Suche sofort um), und weil `einzelne_setzen` viele Felder auf einmal verändert, arbeiten wir mit einer **Kopie** des Rasters je Knoten. Dann gibt es nichts zurückzunehmen. Jetzt die Tabelle mit allen Verfahren:

In [ ]:
tabelle([('erstes leeres Feld', wahl_erstes, False),
         ('wenigste Kandidaten', wahl_wenigste, False),
         ('wenigste + erzwungene Felder', wahl_wenigste, True),
         ('erstes + erzwungene Felder', wahl_erstes, True)])

Verfahren                         leicht    mittel    schwer    brutal
erstes leeres Feld                 4 209    19 023  > 50 000  > 50 000
wenigste Kandidaten                   52       117       482    45 268
wenigste + erzwungene Felder           1        16       131    12 095
erstes + erzwungene Felder             1        35     3 012  > 50 000


**Lies die Tabelle:** Mit erzwungenen Feldern braucht `leicht` nur noch **1 Knoten** (die Logik allein löst es, es muss nie geraten werden), `mittel` 16, `schwer` 131 und selbst das „brutale“ Sudoku **12 095 Knoten**. Das Verfahren mit **beiden** Ideen ist hier das beste: Vergleiche die dritte mit der vierten Zeile (erstes leeres Feld, aber mit erzwungenen Feldern): `brutal` braucht dort > 50 000 Knoten, `schwer` 3 012.

| Idee | Wirkung |
|---|---|
| Wo man verzweigt | verkleinert den Baum **um Größenordnungen** |
| Vorausschauend prüfen | erkennt Widersprüche **ohne zu raten**, die Knoten sind aber teurer |

#### Frage 1.7

1. Mit `wahl_wenigste` und `einzelne_setzen` braucht `brutal` immer noch 12 095 Knoten. Man könnte noch weitere Regeln einbauen (zum Beispiel: Eine Ziffer, die in einem Block nur auf **ein** Feld passt, muss dorthin). Wann lohnt sich eine solche Regel, und wie misst du es?
2. Wir haben jeden Knoten gezählt, nicht die Zeit. Nenne einen Grund, warum die Zahl der Knoten allein nicht reicht, um Verfahren zu vergleichen.

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Eine Regel lohnt sich, wenn sie **viele Knoten spart** und selbst **billig** ist (oder weniger kostet, als sie spart). Du misst es mit **derselben Tabelle**: Verfahren ohne und mit der Regel auf mehreren Beispielen, Knoten zählen **und** die Kosten pro Knoten abschätzen (wie oft wird `kandidaten` aufgerufen?). Eine Regel, die nur selten greift, kostet mehr, als sie bringt.
2. Die Knoten sind **verschieden teuer**: Ein Knoten von `suche_voraus` kostet viel mehr als einer von `suche` (er setzt viele Felder und prüft alle leeren). Gleiche Knotenzahl heißt nicht gleiche Arbeit. Besser ist ein zweiter Zähler für die **Grundoperationen** (zum Beispiel die Aufrufe von `kandidaten`), so wie wir es in Notebook 04 mit den geprüften Paaren gemacht haben.

</details>

### Eine Lösung, alle Lösungen oder keine?

Es gibt drei verschiedene Fragen:

| Frage | Die Suche hört auf, wenn … | Ergebnis |
|---|---|---|
| Gib mir **eine** Lösung | die erste gefunden ist | Lösung oder „es gibt keine“ |
| **Wie viele** Lösungen gibt es? | alle Zweige durchsucht sind (oder eine Grenze erreicht ist) | eine Zahl |
| Ist die Lösung **eindeutig**? | eine **zweite** Lösung gefunden oder alles durchsucht ist | ja oder nein |

„Es gibt keine Lösung“ **beweist** nur eine **vollständige** Suche, die alle Zweige durchsucht hat, genau das, was die lokale Suche aus Notebook 04 nicht kann. Für die Eindeutigkeit reicht es, bis **2** zu zählen: Danach ist die Antwort klar.

#### Aufgabe 1.8

Schreibe `zaehle_loesungen(raster, grenze)`: Sie zählt die Lösungen und hört auf, sobald `grenze` erreicht ist (dann steht nicht die genaue Zahl, sondern „mindestens `grenze`“). Verwende `einzelne_setzen` und `wahl_wenigste`. Die Zelle zählt für `leicht` und `mittel` bis 5.

**Erwartete Ausgabe:**
```
leicht 1
mittel 1
```

In [ ]:
def zaehle_loesungen(raster, grenze):
    """Zählt die Lösungen, hört aber bei grenze auf. Nutzt wahl_wenigste und einzelne_setzen."""
    # TODO
    return 0


for name in ['leicht', 'mittel']:
    print(name, zaehle_loesungen(lies_sudoku(TEXTE[name]), 5))

<details>
<summary>Tipp anzeigen</summary>

Wie `suche_voraus`, aber statt beim ersten Erfolg zurückzukehren, **addierst** du die Ergebnisse der Aufrufe. Kein leeres Feld mehr: Das ist **eine** Lösung. Gib beim Aufruf `grenze - summe` weiter, dann hört die Suche rechtzeitig auf.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def zaehle_loesungen(raster, grenze):
    """Zählt die Lösungen, hört aber bei grenze auf. Nutzt wahl_wenigste und einzelne_setzen."""
    raster = raster[:]
    if not einzelne_setzen(raster):
        return 0
    feld = wahl_wenigste(raster)
    if feld is None:
        return 1
    summe = 0
    for ziffer in kandidaten(raster, feld):
        raster[feld] = ziffer
        summe += zaehle_loesungen(raster, grenze - summe)
        if summe >= grenze:
            break
    return summe


for name in ['leicht', 'mittel']:
    print(name, zaehle_loesungen(lies_sudoku(TEXTE[name]), 5))
```

</details>

In [ ]:
# mittel ohne die Vorgabe auf Feld 15, und leicht mit einer falschen 1 auf Feld 2
mehrdeutig = lies_sudoku(TEXTE['mittel'])
mehrdeutig[15] = 0
unloesbar = lies_sudoku(TEXTE['leicht'])
unloesbar[2] = 1
print('mehrdeutig', zaehle_loesungen(mehrdeutig, 10))
print('unlösbar  ', zaehle_loesungen(unloesbar, 10))
print('die 1 auf Feld 2 verletzt keine Regel:', 1 in kandidaten(lies_sudoku(TEXTE['leicht']), 2))

mehrdeutig 3
unlösbar   0
die 1 auf Feld 2 verletzt keine Regel: True


`leicht` und `mittel` sind **eindeutig** (eine Lösung). Wenn man bei `mittel` die Vorgabe auf Feld 15 weglässt, gibt es **3** Lösungen. Und setzt man bei `leicht` auf das Feld 2 eine 1, verletzt das keine Regel direkt (die 1 ist ein Kandidat), und trotzdem gibt es **keine Lösung**: Das hat die vollständige Suche **bewiesen**.

### Finde den Fehler

Diese Suche hat dieselbe Idee wie `suche`, liefert aber bei `leicht` das falsche Ergebnis.

In [ ]:
def suche_fehler(raster, wahl, zaehler):
    zaehler['knoten'] += 1
    feld = wahl(raster)
    if feld is None:
        return raster[:]
    for ziffer in kandidaten(raster, feld):
        raster[feld] = ziffer
        ergebnis = suche_fehler(raster, wahl, zaehler)
        if ergebnis is not None:
            return ergebnis
    return None


vorgabe = lies_sudoku(TEXTE['leicht'])
zaehler = neuer_zaehler()
ergebnis = suche_fehler(vorgabe[:], wahl_erstes, zaehler)
print('Lösung gefunden' if ergebnis is not None else 'keine Lösung gefunden', ',', zaehler['knoten'], 'Knoten')

keine Lösung gefunden , 25 Knoten


#### Frage 1.9

Die Suche meldet „keine Lösung“, obwohl `leicht` lösbar ist. Was ist falsch, und wie hast du es gefunden? Wie hätte ein **Schiedsrichter** das Problem bemerkt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

Es fehlt das **Zurücknehmen**: Nach dem Aufruf bleibt die Ziffer im Raster stehen (`raster[feld] = 0` fehlt). Probiert die Suche die nächste Ziffer, stehen die alten Ziffern noch im Raster, und die Kandidaten der Nachbarn sind falsch. Außerdem muss sie in jedem Zweig **dieselbe** Ausgangslage haben. Den Fehler findet man am besten, wenn man das Raster nach einem erfolglosen Aufruf **ausgibt** oder die Zahl der Knoten mit der richtigen Version vergleicht (hier viel zu klein: Die Suche gibt viel zu früh auf). Ein Schiedsrichter prüft nur gefundene Lösungen: Er hätte bei **„keine Lösung“** nichts zu prüfen und wäre blind. Deshalb hilft hier ein Test mit **bekannten lösbaren** Beispielen.

</details>

## 2. Pentominos: Symmetrien und Teilzustände merken

Ein **Pentomino** besteht aus fünf Quadraten. Es gibt zwölf verschiedene, sie heißen nach ihrer Form `F`, `I`, `L`, `N`, `P`, `T`, `U`, `V`, `W`, `X`, `Y`, `Z`. Gedreht und gespiegelt zählt ein Pentomino als dasselbe Teil. **Aufgabe:** Fülle ein Rechteck lückenlos mit **verschiedenen** Pentominos (jedes höchstens einmal). Wir nehmen ein Rechteck aus 5 Zeilen und 6 Spalten (30 Felder, also sechs Teile von zwölf möglichen).

`lagen_von` berechnet alle Lagen eines Teils (das X hat nur eine, das F acht), `platzierungen(h, b)` liefert für jedes Teil alle Möglichkeiten, es in das Rechteck zu legen, jeweils als `frozenset` der Feldnummern (Zeile · b + Spalte). Ein `frozenset` ist eine Menge, die sich nicht mehr ändert. Deshalb kann man sie als **Schlüssel** in einem Wörterbuch oder in einer Menge verwenden. Mit `&` bildet man den Durchschnitt zweier Mengen, mit `|` die Vereinigung.

In [ ]:
FORMEN = {
    'F': ['.##', '##.', '.#.'], 'I': ['#####'], 'L': ['####', '#...'], 'N': ['###.', '..##'],
    'P': ['##', '##', '#.'], 'T': ['###', '.#.', '.#.'], 'U': ['#.#', '###'], 'V': ['#..', '#..', '###'],
    'W': ['#..', '##.', '.##'], 'X': ['.#.', '###', '.#.'], 'Y': ['####', '.#..'], 'Z': ['##.', '.#.', '.##'],
}


def lagen_von(form):
    """Alle verschiedenen Lagen (drehen und spiegeln) einer Form, als Liste von Zellenlisten (Zeile, Spalte) ab (0, 0)."""
    zellen = []
    for zeile in range(len(form)):
        for spalte in range(len(form[zeile])):
            if form[zeile][spalte] == '#':
                zellen.append((zeile, spalte))
    ergebnis = []
    for spiegeln in range(2):
        for drehungen in range(4):
            neu = []
            for z, s in zellen:
                neu.append((s, -z))                        # eine Vierteldrehung
            zellen = neu
            z0 = min(z for z, s in zellen)
            s0 = min(s for z, s in zellen)
            lage = sorted((z - z0, s - s0) for z, s in zellen)
            if lage not in ergebnis:
                ergebnis.append(lage)
        zellen = [(z, -s) for z, s in zellen]               # spiegeln
    return ergebnis


def platzierungen(h, b):
    """Für jedes Teil alle Möglichkeiten, es in ein Rechteck mit h Zeilen und b Spalten zu legen.
    Eine Möglichkeit ist die Menge der Feldnummern (9 * Zeile + Spalte, hier Zeile * b + Spalte)."""
    plaetze = {}
    for name in FORMEN:
        plaetze[name] = []
        for lage in lagen_von(FORMEN[name]):
            for z0 in range(h):
                for s0 in range(b):
                    felder = []
                    for z, s in lage:
                        if z0 + z < h and s0 + s < b:
                            felder.append((z0 + z) * b + s0 + s)
                    if len(felder) == 5:
                        plaetze[name].append(frozenset(felder))
    return plaetze


print({name: len(lagen_von(FORMEN[name])) for name in FORMEN})
print('Möglichkeiten, das X in 5 x 6 zu legen:', len(platzierungen(5, 6)['X']))
print('Möglichkeiten, das F in 5 x 6 zu legen:', len(platzierungen(5, 6)['F']))

{'F': 8, 'I': 2, 'L': 8, 'N': 8, 'P': 8, 'T': 4, 'U': 4, 'V': 4, 'W': 4, 'X': 1, 'Y': 8, 'Z': 4}
Möglichkeiten, das X in 5 x 6 zu legen: 12
Möglichkeiten, das F in 5 x 6 zu legen: 96


### Die Suche: erstes freies Feld oder Feld mit den wenigsten Möglichkeiten

Der Suchbaum entsteht so: Wähle ein **freies Feld**. Jedes Teil, das noch nicht benutzt ist und das **dieses Feld abdeckt**, ohne ein belegtes Feld zu überdecken, ist eine Möglichkeit. Das ist dieselbe Struktur wie beim Sudoku: Das Feld ist die Entscheidung, die Möglichkeiten sind die Kandidaten. Wieder gibt es zwei Auswahlregeln: das **erste** freie Feld und das freie Feld mit den **wenigsten** Möglichkeiten. Die Funktion `fuelle` zählt **alle** Füllungen und die Knoten. Die Parameter `x_zuerst`, `x_links` und `merken` brauchen wir erst weiter unten.

In [ ]:
def fuelle(h, b, wahl, x_zuerst=False, x_links=False, merken=False):
    """Zählt, wie viele Füllungen des Rechtecks h x b mit verschiedenen Pentominos es gibt (jedes Teil höchstens einmal).
    wahl: 'erste' = erstes freies Feld, 'wenigste' = freies Feld mit den wenigsten Möglichkeiten.
    x_zuerst: Das X wird zuerst gelegt (es kommt dann in jeder gezählten Füllung vor).
    x_links: Das X darf nur mit dem Mittelpunkt in der linken Hälfte liegen (Symmetrie).
    merken: Teilzustände merken. Gibt ein Wörterbuch mit loesungen, knoten, treffer, mit_x zurück."""
    plaetze = platzierungen(h, b)
    if x_links:
        links = []
        for felder in plaetze['X']:
            if sorted(felder)[2] % b <= (b - 1) // 2:           # Spalte des Mittelpunkts
                links.append(felder)
        plaetze['X'] = links
    nach_feld = {}                                           # Feld -> Liste von (Name, Felder), die es abdecken
    for name in plaetze:
        for felder in plaetze[name]:
            for f in felder:
                nach_feld.setdefault(f, []).append((name, felder))
    alle = frozenset(range(h * b))
    st = {'loesungen': 0, 'knoten': 0, 'treffer': 0, 'mit_x': 0}
    gemerkt = {}

    def moeglichkeiten(f, belegt, benutzt):
        ergebnis = []
        for name, felder in nach_feld[f]:
            if name not in benutzt and not (felder & belegt):
                ergebnis.append((name, felder))
        return ergebnis

    def rek(belegt, benutzt):
        """Zahl der Füllungen, die sich an diesen Zustand anschließen."""
        st['knoten'] += 1
        if belegt == alle:
            st['mit_x'] += 'X' in benutzt
            return 1
        schluessel = (belegt, benutzt)
        if merken and schluessel in gemerkt:
            st['treffer'] += 1
            return gemerkt[schluessel]
        frei = sorted(alle - belegt)
        if wahl == 'erste':
            optionen = moeglichkeiten(frei[0], belegt, benutzt)
        else:
            optionen = None
            for f in frei:
                kandidaten_f = moeglichkeiten(f, belegt, benutzt)
                if optionen is None or len(kandidaten_f) < len(optionen):
                    optionen = kandidaten_f
                    if len(optionen) == 0:
                        break
        summe = 0
        for name, felder in optionen:
            summe += rek(belegt | felder, benutzt | {name})
        if merken:
            gemerkt[schluessel] = summe
        return summe

    if x_zuerst:
        for felder in plaetze['X']:
            st['loesungen'] += rek(felder, frozenset(['X']))
    else:
        st['loesungen'] = rek(frozenset(), frozenset())
    return st

In [ ]:
for wahl in ['erste', 'wenigste']:
    st = fuelle(5, 6, wahl)
    print(f"{wahl:<9} {st['loesungen']} Füllungen, {st['knoten']:>6} Knoten")

erste     2164 Füllungen,  80747 Knoten
wenigste  2164 Füllungen,  23514 Knoten


Beide Regeln finden dieselben **2164 Füllungen** (das ist eine Kontrolle, dass wir nichts verloren haben), aber das Feld mit den wenigsten Möglichkeiten braucht nur **23 514** statt **80 747** Knoten. Auch hier gilt: Die Reihenfolge der Entscheidungen bestimmt die Größe des Baums.

#### Frage 2.1

1. Warum findet man bei „erstes freies Feld“ in einem Rechteck immer eine Möglichkeit, in der ein Teil **genau dieses Feld** abdeckt? Warum verliert man keine Füllung, wenn man nur Teile betrachtet, die das gewählte Feld abdecken?
2. Was wäre die Alternative, und warum wäre sie viel größer? (Tipp: Man könnte zuerst ein **Teil** wählen und dann seine Lage.)

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Jede Füllung muss das gewählte freie Feld mit **irgendeinem** Teil abdecken. Jede Füllung gehört also in **genau einen** Zweig, nämlich den des Teils, das dieses Feld abdeckt. Wir verlieren keine Füllung und zählen keine doppelt. Das ist das Prinzip der **vollständigen Fallunterscheidung**: Die Möglichkeiten an einem Knoten müssen alle Lösungen abdecken und sich nicht überschneiden.
2. Wählt man zuerst ein Teil und dann seine Lage, entscheidet man in der **Reihenfolge der Teile**, nicht der Felder. Dann hängen die gelegten Teile in der Luft (irgendwo im Rechteck), und Lücken, die sich nie mehr füllen lassen, bemerkt man spät. Derselbe Füllungsbaum würde auch **jede Füllung mehrfach** zählen, je nachdem, in welcher Reihenfolge man die Teile nimmt (6 Teile: 720 Reihenfolgen). Bei „Feld zuerst“ legt man die Teile immer in einer **festen Reihenfolge der Felder**, das schneidet diese Mehrfachzählung weg.

</details>

### Teilzustände merken

Zwei verschiedene Wege können zum **gleichen Zustand** führen: dieselben belegten Felder, dieselben benutzten Teile (zum Beispiel, wenn zwei Teile zusammen das gleiche Gebiet auf zwei Arten füllen). Dann ist alles, was danach kommt, gleich. Wir merken uns für jeden Zustand (ein Paar aus `frozenset` der belegten Felder und `frozenset` der benutzten Teile) die **Zahl der Füllungen**, die sich anschließen, in einem Wörterbuch `gemerkt`. Trifft die Suche den Zustand wieder, nimmt sie die Zahl aus dem Wörterbuch, statt den Teilbaum noch einmal zu durchsuchen (**Memoisierung**, [learn_bwinf1, Notebook 10](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/10_merken.ipynb)).

In [ ]:
for wahl in ['erste', 'wenigste']:
    st = fuelle(5, 6, wahl, merken=True)
    print(f"{wahl:<9} {st['loesungen']} Füllungen, {st['knoten']:>6} Knoten, {st['treffer']} Treffer im Merkzettel")

erste     2164 Füllungen,  71909 Knoten, 7768 Treffer im Merkzettel
wenigste  2164 Füllungen,  21064 Knoten, 856 Treffer im Merkzettel


Die Füllungen sind dieselben (2164), und es gibt tatsächlich Treffer im Merkzettel: bei „erstes freies Feld“ 7 768 Treffer, die Knoten sinken von 80 747 auf 71 909, bei „wenigste Möglichkeiten“ von 23 514 auf 21 064. Das ist **nicht viel**: Beim Zählen **aller** Füllungen gleichen sich nur wenige Zustände. Wie stark das Merken wirken kann, siehst du gleich bei Konfetti, wo gleiche Teilzustände sehr oft vorkommen.

### Symmetrien: nur einen von vielen gleichen Zweigen suchen

Dreht man das Rechteck um 180° oder spiegelt man es, bleibt jede Füllung eine Füllung. Die Füllungen kommen also in **Gruppen gleicher Füllungen** vor, und es genügt, **eine pro Gruppe** zu suchen. Man braucht einen Trick, der die Spiegelbilder **früh** ausschließt, nicht erst am Ende. Hier: Das **X** ist das einzige Teil, das sich beim Drehen und Spiegeln nicht ändert, und es liegt mit seinem Mittelpunkt entweder in der linken oder in der rechten Hälfte. Eine Spiegelung links-rechts tauscht die beiden Hälften. Wenn wir das X **zuerst** legen und dem X nur die linke Hälfte erlauben, sparen wir die Hälfte der Füllungen mit X ein.

Dazu zwei Schritte in `fuelle`: Mit `x_zuerst=True` wird zuerst das X gelegt (es kommt dann in **jeder** gezählten Füllung vor). Mit `x_links=True` darf sein Mittelpunkt nur in den linken Spalten liegen.

In [ ]:
voll = fuelle(5, 6, 'wenigste')
print('alle Füllungen:', voll['loesungen'], ', davon mit dem X:', voll['mit_x'])
for x_links in [False, True]:
    st = fuelle(5, 6, 'wenigste', x_zuerst=True, x_links=x_links)
    print(f"X zuerst{', X links' if x_links else ''}: {st['loesungen']} Füllungen, {st['knoten']} Knoten")

alle Füllungen: 2164 , davon mit dem X: 136
X zuerst: 136 Füllungen, 1058 Knoten
X zuerst, X links: 68 Füllungen, 529 Knoten


Von den 2164 Füllungen enthalten **136** das X. Legt man das X zuerst, findet die Suche genau diese 136 mit nur **1 058** Knoten (statt 23 514 für alle). Erlaubt man dem X nur die linke Hälfte, sind es **68** Füllungen, genau die **Hälfte**, mit nur **529** Knoten. Die andere Hälfte sind die Spiegelbilder, die wir nicht zu suchen brauchen: 68 · 2 = 136. Das ist die Symmetrie-Regel, **vorher gewusst, nicht nachträglich aussortiert**.

#### Frage 2.2

1. Warum lässt sich die Symmetrie mit dem **X** so leicht brechen, mit dem **F** aber nicht?
2. Welche Symmetrien hat ein Rechteck aus 5 × 6 Feldern? Warum würde „X nur in der oberen Hälfte“ (statt links) die Zahl der Füllungen **nicht** genau halbieren?
3. Du suchst ein 6 × 10-Rechteck mit **allen** zwölf Teilen. Wie viele Füllungen gibt es, wenn es insgesamt N Gruppen von Spiegelungen und Drehungen gibt, in denen keine Füllung sich selbst entspricht?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Das X sieht in **allen** acht Lagen gleich aus. Wenn man das Rechteck spiegelt, liegt das X danach **eindeutig** woanders (sein Mittelpunkt wird gespiegelt), und man kann sich einfach auf eine Hälfte festlegen. Das F hat acht Lagen, die sich unterscheiden: Spiegelt man das Rechteck, ändert sich auch die Lage des F, und man müsste auch **diese** Lage festlegen. Es geht, aber es ist komplizierter.
2. Ein Rechteck hat vier Symmetrien: nichts tun, links-rechts spiegeln, oben-unten spiegeln und um 180° drehen. Die Zeilen 0 bis 4 haben eine **mittlere** Zeile (Nummer 2), die sich beim Spiegeln oben-unten **nicht** ändert. Ein X mit Mittelpunkt in der mittleren Zeile bleibt in der „oberen Hälfte“ und seine Spiegelung auch: Beide würden gezählt. Die Spalten 0 bis 5 haben keine mittlere Spalte, links-rechts spiegeln tauscht die Hälften sauber.
3. Es gibt 4 · N Füllungen: Jede Gruppe hat vier verschiedene Füllungen (Original und die drei Spiegelungen beziehungsweise Drehungen). Die 6 × 10-Aufgabe hat bekanntlich 2 339 Gruppen, das sind 9 356 Füllungen. Die Suche mit Symmetrie sucht nur ein Viertel.

</details>

## Wettbewerbs-Ecke 1: Konfetti

*Angelehnt an Aufgabe 3 „Konfetti“ der 2. Runde des 43. Bundeswettbewerbs Informatik (2024/25). Vereinfacht: keine unleserlichen Einträge, Annas eigene Zeile ist unbekannt, und es geht nur um Teil a (gibt es eine zulässige Anordnung?).*

Auf einem Aushang trägt jede Person in **ihre Zeile** ein, an welchen Terminen (**Spalten**) sie kann (`y`) oder nicht kann (`n`). Der kleine Bruder hat den Aushang in **Spalten** zerschnitten und durcheinandergebracht. Eine Anordnung der Spalten ist **zulässig**, wenn in **jeder Zeile alle `y` zusammenhängend** stehen (zwischen zwei `y` also kein `n`). Gibt es eine zulässige Anordnung, und wie sieht sie aus?

Wir speichern jede Spalte als Zeichenkette von oben nach unten (`'nyn'`), eine Anordnung ist eine Liste von Spalten. Der Schiedsrichter `ist_zulaessig` prüft sie. Das Probieren aller Reihenfolgen scheidet aus: Bei 17 Spalten wären es 17! ≈ 355 Billionen. Wir legen die Spalten **von links nach rechts** (Backtracking) und führen den Suchbaum mit den Ideen aus diesem Notebook klein.

In [ ]:
def lies_aushang(dateiname):
    """Liest einen Aushang (ohne ? und ohne Annas Zeile) und gibt die Spalten als Zeichenketten aus 'y' und 'n' zurück."""
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    kopf = zeilen[0].split()
    r = int(kopf[0])
    c = int(kopf[1])
    tabelle = []
    for i in range(r):
        tabelle.append(zeilen[1 + i].split())
    spalten = []
    for j in range(c):
        spalte = ''
        for i in range(r):
            spalte = spalte + tabelle[i][j]
        spalten.append(spalte)
    return spalten


def zeile_ist_ok(zeile):
    """Zwischen dem ersten und dem letzten 'y' darf kein 'n' stehen."""
    if 'y' not in zeile:
        return True
    erstes = zeile.index('y')
    letztes = zeile.rindex('y')
    return 'n' not in zeile[erstes:letztes + 1]


def ist_zulaessig(anordnung):
    """Schiedsrichter: Ist in jeder Zeile der Anordnung das 'y' zusammenhängend?"""
    for z in range(len(anordnung[0])):
        zeile = ''
        for spalte in anordnung:
            zeile = zeile + spalte[z]
        if not zeile_ist_ok(zeile):
            return False
    return True


klein = ['nyn', 'yyy', 'nny', 'yny']                       # vier Spalten mit drei Zeilen
print(ist_zulaessig(klein), ist_zulaessig([klein[0], klein[1], klein[3], klein[2]]))

### Regel 1: Man darf keinen angefangenen Block zerreißen

Während wir von links nach rechts legen, merken wir uns für jede Zeile `z`, wie viele `y` schon liegen (`gelegt[z]`) und wie viele **noch fehlen** (`rest[z]`). Ist `gelegt[z] > 0` **und** `rest[z] > 0`, dann **läuft** in Zeile `z` ein Block: Er hat angefangen und es kommen noch `y`. Eine Spalte mit einem `n` in dieser Zeile würde den Block zerreißen, die späteren `y` lägen nicht mehr zusammen. Diese Regel ist **exakt**: Eine Anordnung ist genau dann zulässig, wenn jede Spalte beim Legen diese Regel erfüllt hat.

#### Aufgabe E1.1

Schreibe `darf_kommen(spalte, gelegt, rest)`. Sie sagt, ob die Spalte als nächste gelegt werden darf (**Regel 1**). Die Zelle probiert zwei Spalten für vier Zeilen aus: In Zeile 0 läuft ein Block (ein `y` liegt, eins fehlt), Zeile 1 ist fertig, Zeile 2 und 3 sind unberührt beziehungsweise ein `y` fehlt noch.

**Erwartete Ausgabe:**
```
True
False
```

In [ ]:
def darf_kommen(spalte, gelegt, rest):
    """gelegt[z]: Wie viele 'y' liegen in Zeile z schon? rest[z]: Wie viele 'y' fehlen noch?
    Darf die Spalte als nächste gelegt werden, ohne in einer Zeile einen angefangenen Block zu zerreißen?"""
    # TODO
    return True


# vier Zeilen: In Zeile 0 liegt schon ein 'y' und eins fehlt noch, Zeile 1 ist fertig, Zeile 2 nicht angefangen
print(darf_kommen('ynyn', [1, 2, 0, 0], [1, 0, 2, 1]))
print(darf_kommen('nyyn', [1, 2, 0, 0], [1, 0, 2, 1]))

<details>
<summary>Tipp anzeigen</summary>

Gehe alle Zeilen durch und gib `False` zurück, wenn in einer Zeile ein Block läuft und die Spalte dort kein `y` hat.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def darf_kommen(spalte, gelegt, rest):
    """gelegt[z]: Wie viele 'y' liegen in Zeile z schon? rest[z]: Wie viele 'y' fehlen noch?
    Darf die Spalte als nächste gelegt werden, ohne in einer Zeile einen angefangenen Block zu zerreißen?"""
    for z in range(len(spalte)):
        if gelegt[z] > 0 and rest[z] > 0 and spalte[z] != 'y':          # In Zeile z läuft ein Block, die Spalte hat ein 'n'
            return False
    return True


# vier Zeilen: In Zeile 0 liegt schon ein 'y' und eins fehlt noch, Zeile 1 ist fertig, Zeile 2 nicht angefangen
print(darf_kommen('ynyn', [1, 2, 0, 0], [1, 0, 2, 1]))
print(darf_kommen('nyyn', [1, 2, 0, 0], [1, 0, 2, 1]))
```

</details>

### Die Suche und ihre Verbesserungen

Jetzt die Suche. Sie hat drei **Schalter**, die wir einzeln ein- und ausschalten und mit einer Tabelle der **Knoten** messen:

1. **`typen`: gleiche Spalten zusammenfassen.** Gibt es mehrere gleiche Spalten, probiert die Suche nicht jede einzeln als „nächste“: Sie probiert jede **Art** einmal und merkt sich, wie viele von ihr noch übrig sind. Das ist das Muster „gleiche Teile zusammenfassen“: Sonst werden gleiche Anordnungen mehrfach gesucht.
2. **`merken`: Sackgassen merken.** Was schon liegt, bestimmt `gelegt` und `rest` vollständig, und damit, was danach möglich ist. Der **Zustand** ist also, **wie oft jede Art schon liegt** (ein Tupel `tuple(benutzt)`), nicht die Reihenfolge, in der sie lagen. Findet die Suche von einem Zustand aus keine Lösung, schreibt sie ihn in die Menge `sackgassen`. Kommt sie später **auf anderem Weg** zum selben Zustand, kehrt sie sofort um.
3. **`blockade`: zwei Zeilen blockieren sich.** Angenommen, in den Zeilen `a` und `b` läuft ein Block (oder beginnt mit der Spalte, die wir legen wollen). Gibt es noch eine Spalte X mit `y` in `a` und `n` in `b`, **und** eine Spalte Y mit `y` in `b` und `n` in `a`, dann kann keine von beiden je gelegt werden: X muss warten, bis Zeile `b` fertig ist, und Y, bis Zeile `a` fertig ist, aber Zeile `a` wird erst mit X und Zeile `b` erst mit Y fertig. Das ist eine **Blockade**, und die Sackgasse ist **schon jetzt** sicher, nicht erst nach vielen weiteren Spalten.

In [ ]:
def suche_konfetti(spalten, typen=True, merken=True, blockade=True, grenze=20000):
    """Legt die Spalten von links nach rechts. Gibt (Anordnung oder None, Zahl der Knoten, abgebrochen) zurück.
    typen: gleiche Spalten zusammenfassen. merken: Sackgassen merken. blockade: die Blockaderegel aus Abschnitt 3."""
    r = len(spalten[0])
    if typen:
        arten = sorted(set(spalten))
        anzahl = []
        for art in arten:
            anzahl.append(spalten.count(art))
    else:
        arten = spalten[:]
        anzahl = [1] * len(arten)
    rest = [0] * r                                    # so viele 'y' liegen in Zeile z noch nicht
    for spalte in spalten:
        for z in range(r):
            if spalte[z] == 'y':
                rest[z] += 1
    gelegt = [0] * r
    benutzt = [0] * len(arten)                        # wie oft jede Art schon gelegt ist
    folge = []
    sackgassen = set()
    stand = {'knoten': 0, 'abgebrochen': False}

    def blockiert(a, b, t):
        """Gibt es (ohne die Art t, die wir gerade legen) noch eine Spalte mit y in a und n in b UND eine mit y in b und n in a?"""
        x = False
        y = False
        for u in range(len(arten)):
            frei = anzahl[u] - benutzt[u]
            if u == t:
                frei -= 1
            if frei > 0:
                if arten[u][a] == 'y' and arten[u][b] == 'n':
                    x = True
                if arten[u][b] == 'y' and arten[u][a] == 'n':
                    y = True
        return x and y

    def rek(n):
        stand['knoten'] += 1
        if stand['knoten'] > grenze:
            stand['abgebrochen'] = True
            return False
        if n == len(spalten):
            return True
        schluessel = tuple(benutzt)                   # was schon liegt, bestimmt gelegt und rest
        if merken and schluessel in sackgassen:
            return False
        for t in range(len(arten)):
            if benutzt[t] == anzahl[t]:
                continue
            spalte = arten[t]
            if not darf_kommen(spalte, gelegt, rest):
                continue
            if blockade:
                neu = []
                laufend = []
                for z in range(r):
                    if spalte[z] == 'y' and gelegt[z] == 0:
                        neu.append(z)
                    if gelegt[z] > 0 or spalte[z] == 'y':
                        laufend.append(z)
                if any(blockiert(a, b, t) for a in neu for b in laufend):
                    continue
            for z in range(r):
                if spalte[z] == 'y':
                    gelegt[z] += 1
                    rest[z] -= 1
            benutzt[t] += 1
            folge.append(t)
            if rek(n + 1):
                return True
            if stand['abgebrochen']:
                return False
            folge.pop()
            benutzt[t] -= 1
            for z in range(r):
                if spalte[z] == 'y':
                    gelegt[z] -= 1
                    rest[z] += 1
        if merken:
            sackgassen.add(schluessel)
        return False

    gefunden = rek(0)
    anordnung = None
    if gefunden:
        anordnung = []
        for t in folge:
            anordnung.append(arten[t])
    return anordnung, stand['knoten'], stand['abgebrochen']

In [ ]:
anordnung, knoten, abgebrochen = suche_konfetti(klein)
print(anordnung, knoten, 'Knoten, zulässig:', ist_zulaessig(anordnung))
anordnung, knoten, abgebrochen = suche_konfetti(['yny', 'yyn', 'nyy'])      # drei Zeilen {a, b}, {b, c}, {a, c}
print(anordnung, knoten, 'Knoten')

['nny', 'yny', 'yyy', 'nyn'] 5 Knoten, zulässig: True
None 1 Knoten


Die Dateien `konfetti_mini1` bis `konfetti_mini5` in `daten/` sind kleine Aushänge: `mini1` (7 Zeilen, 17 Spalten) und `mini2` (5 Zeilen, 9 Spalten) sind **Beispiele des Wettbewerbs** (`konfetti04` und `konfetti02` auf den BwInf-Seiten), `mini3` bis `mini5` habe ich selbst erzeugt: Aus zufälligen Zeilen mit zusammenhängenden `y` entstand ein lösbarer Aushang (`mini3`) beziehungsweise, durch Ändern **eines** Eintrags, ein Aushang, der meist nicht mehr lösbar ist (`mini4`, `mini5`). Die Tabelle zeigt die Knoten je Variante (die Grenze ist 20 000):

In [ ]:
VARIANTEN = [('einzelne Spalten', False, False, False),
             ('gleiche Spalten zusammen', True, False, False),
             ('+ Sackgassen merken', True, True, False),
             ('+ Blockaderegel', True, True, True)]
DATEIEN = ['konfetti_mini1', 'konfetti_mini2', 'konfetti_mini3', 'konfetti_mini4', 'konfetti_mini5']


def zeige_konfetti_tabelle():
    print(f'{"Spalten x Zeilen":<28}' + ''.join(f'{lies_groesse(d):>10}' for d in DATEIEN))
    for titel, typen, merken, blockade in VARIANTEN:
        zeile = f'{titel:<28}'
        for d in DATEIEN:
            anordnung, knoten, abgebrochen = suche_konfetti(lies_aushang('daten/' + d + '.txt'), typen, merken, blockade)
            zeile = zeile + f"{('> 20 000' if abgebrochen else Z(knoten)):>10}"
        print(zeile)
    zeile = f'{"lösbar?":<28}'
    for d in DATEIEN:
        anordnung, knoten, abgebrochen = suche_konfetti(lies_aushang('daten/' + d + '.txt'))
        zeile = zeile + f"{('ja' if anordnung else 'nein'):>10}"
    print(zeile)


def lies_groesse(d):
    spalten = lies_aushang('daten/' + d + '.txt')
    return f'{len(spalten)}x{len(spalten[0])}'


def Z(n):
    return f'{n:,}'.replace(',', ' ')


zeige_konfetti_tabelle()

Spalten x Zeilen                  17x7       9x5     16x10     16x10     18x12
einzelne Spalten                    50        86       120  > 20 000  > 20 000
gleiche Spalten zusammen            18        57        38     5 408     3 514
+ Sackgassen merken                 18        57        38       247       541
+ Blockaderegel                     18        15        19       147       277
lösbar?                             ja      nein        ja      nein      nein


**Lies die Tabelle** (die Ergebnisse stimmen mit dem Lösungsnotebook zu Konfetti überein: `mini1` ist lösbar mit 18 Knoten, `mini2` nicht lösbar mit 57 Knoten):

- **Gleiche Spalten zusammenfassen** spart bei `mini1` von 50 auf 18 Knoten und bei `mini2` von 86 auf 57. Bei den selbst erzeugten Beispielen, die gleiche Spalten enthalten, wirkt es noch deutlicher.
- **Sackgassen merken** ist bei den **lösbaren** Beispielen kaum nötig (die Regel 1 allein führt fast ohne Umwege zur Lösung), aber bei den **nicht lösbaren** Beispielen `mini4` und `mini5` entscheidend: von 5 408 und 3 514 auf 247 und 541 Knoten.
- Die **Blockaderegel** schneidet die Sackgassen noch **früher** ab: 147 und 277 Knoten.

**Wer beweisen muss, dass es keine Lösung gibt**, durchsucht den **ganzen** Baum, deshalb entscheiden Merken und frühes Abschneiden. Bei einer lösbaren Aufgabe hat man oft Glück und findet die Lösung früh.

#### Frage E1.2

1. Warum darf der Schlüssel im Merkzettel die **Reihenfolge** der gelegten Spalten vergessen und nur zählen, **wie oft jede Art liegt**?
2. Warum bleibt die Antwort „lösbar oder nicht“ richtig, wenn die Suche einen Zustand schon einmal als Sackgasse erkannt hat und sofort umkehrt?
3. Die echte Aufgabe hat Beispiele mit über 1 000 Spalten. Pro Knoten läuft `darf_kommen` über alle Zeilen, und die Suche probiert alle Arten. Welche Laufzeit pro Knoten bedeutet das, grob in `r` Zeilen und `a` Arten? Warum ist das trotzdem machbar?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Regel 1 hängt nur von `gelegt` und `rest` ab, und die sind **für jede Zeile eine Summe** über die gelegten Spalten, also unabhängig von der Reihenfolge. Zwei verschiedene Reihenfolgen mit gleichen Zählern führen zu **demselben** Zustand, und die Zukunft ist dieselbe.
2. Ob sich ein Zustand zu einer Lösung vervollständigen lässt, hängt **nur vom Zustand** ab, nicht davon, wie man hingekommen ist. Wenn man von ihm aus einmal **alles** durchsucht und nichts gefunden hat, findet man auch beim zweiten Mal nichts. Dabei muss die Suche die Sackgasse erst **eintragen, wenn der ganze Teilbaum durchsucht wurde** (am Ende der Schleife), nicht schon vorher.
3. Pro Knoten ungefähr `a · r` Schritte (jede Art, jede Zeile). Bei 1 000 Spalten und 100 Zeilen sind das etwa 100 000 Schritte pro Knoten. Weil die Regel 1 so streng ist, braucht die Suche bei lösbaren Aufgaben nur **wenig mehr Knoten als Spalten** (das Lösungsnotebook zählt für das größte Beispiel 1 021 Knoten bei 1 020 Spalten), das sind rund 100 Millionen Schritte: für Python an der Grenze. Wer es schneller braucht, lässt `darf_kommen` nur über die **laufenden Zeilen** gehen, statt jedes Mal alle Zeilen durchzugehen.

</details>

## Wettbewerbs-Ecke 2 (für Schnelle): Komplementär

*Angelehnt an Aufgabe 1 „Komplementär“ der 2. Runde des 31. Bundeswettbewerbs Informatik (2012/13). Vereinfacht: kleine Puzzles bis 3 × 3, nur das Lösen (kein Erfinden neuer Puzzles).*

Ein Puzzle besteht aus quadratischen Teilen. Jede Seite hat eine **Einbuchtung** (Kleinbuchstabe `a` bis `d`) oder eine **Ausbuchtung** (Großbuchstabe `A` bis `D`). Zwei Seiten passen, wenn die eine der **Großbuchstabe** der anderen ist: `a` passt zu `A`. Die Teile haben keinen aufgedruckten Hintergrund und dürfen **gedreht** werden. Ein Teil ist die Zeichenkette aus vier Seiten (links, oben, rechts, unten), eine Drehung verschiebt die Zeichenkette um eine Stelle. Gesucht sind alle Anordnungen im `m × n`-Rechteck.

Die Suche füllt die Felder **Zeile für Zeile**. Das Feld hat zwei schon gelegte Nachbarn (links und oben), die vorschreiben, **welche** Seiten das neue Teil links und oben haben muss. Das bietet zwei Verbesserungen:

- **Nachschlagen statt Durchprobieren:** Eine Tabelle (Wörterbuch) liefert zu den verlangten Seiten gleich die **passenden** Teile, ohne alle Teile zu prüfen. Wir zählen die **geprüften Teile**, denn es ändert die **Knoten** nicht, nur die **Arbeit pro Knoten**.
- **Symmetrie:** Dreht man eine Lösung um 90°, ist das wieder eine Lösung. Also kommen die Lösungen in **Vierergruppen**. Wir legen das Teil 0 deshalb **nur in einer Lage** fest (in der es in der Datei steht): Das ist eine **Normalform** (wie bei [learn_bwinf1, Notebook 03](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/03_schranken_symmetrien.ipynb)).

In [ ]:
def lies_puzzle(dateiname):
    """Gibt (m, n, teile) zurück. Ein Teil hat vier Zeichen: links, oben, rechts, unten."""
    with open(dateiname, encoding='utf-8') as datei:
        woerter = datei.read().split()
    return int(woerter[0]), int(woerter[1]), woerter[2:]


def drehe(teil, k):
    return teil[k:] + teil[:k]


def lagen(teil):
    """Die verschiedenen Lagen eines Teils."""
    ergebnis = []
    for k in range(4):
        if drehe(teil, k) not in ergebnis:
            ergebnis.append(drehe(teil, k))
    return ergebnis


def passt(seite1, seite2):
    """Eine Ausbuchtung (Großbuchstabe) passt in die Einbuchtung (Kleinbuchstabe) desselben Buchstabens."""
    return seite1 == seite2.swapcase()


def ist_loesung(m, n, teile, raster):
    """Schiedsrichter: raster[(zeile, spalte)] = (Nummer des Teils, Lage). Jedes Teil genau einmal, alle Nachbarn passen."""
    nummern = []
    for z in range(m):
        for s in range(n):
            nr, lage = raster[(z, s)]
            nummern.append(nr)
            if lage not in lagen(teile[nr]):
                return False
            if s > 0 and not passt(raster[(z, s - 1)][1][2], lage[0]):
                return False
            if z > 0 and not passt(raster[(z - 1, s)][1][3], lage[1]):
                return False
    return sorted(nummern) == list(range(len(teile)))


m2, n2, teile2 = lies_puzzle('daten/puzzle_mini1.txt')
print(m2, n2, teile2)

In [ ]:
def suche_puzzle(m, n, teile, nachschlagen, symmetrie, alle):
    """Legt die Felder Zeile für Zeile. Gibt ein Wörterbuch mit loesungen, knoten, geprueft und erste (eine Lösung) zurück.
    nachschlagen: Teile aus einer Tabelle holen statt alle durchzuprüfen. symmetrie: Teil 0 nur in seiner ersten Lage.
    alle: alle Lösungen zählen (sonst Abbruch nach der ersten)."""
    alle_lagen = []
    tabelle = {}
    for nr in range(len(teile)):
        for lage in lagen(teile[nr]):
            alle_lagen.append((nr, lage))
            tabelle.setdefault((lage[0], lage[1]), []).append((nr, lage))    # (links, oben) -> passende Teile
    benutzt = [False] * len(teile)
    raster = {}
    st = {'loesungen': 0, 'knoten': 0, 'geprueft': 0, 'erste': None}

    def kandidaten_fuer(z, s):
        links = raster[(z, s - 1)][1][2].swapcase() if s > 0 else None       # was links gebraucht wird
        oben = raster[(z - 1, s)][1][3].swapcase() if z > 0 else None
        if nachschlagen and links is not None and oben is not None:
            st['geprueft'] += len(tabelle.get((links, oben), []))
            return tabelle.get((links, oben), [])
        ergebnis = []
        for nr, lage in alle_lagen:                                          # sonst alle Teile in allen Lagen prüfen
            st['geprueft'] += 1
            if (links is None or lage[0] == links) and (oben is None or lage[1] == oben):
                ergebnis.append((nr, lage))
        return ergebnis

    def rek(feld):
        st['knoten'] += 1
        if feld == m * n:
            st['loesungen'] += 1
            if st['erste'] is None:
                st['erste'] = dict(raster)
            return not alle
        z, s = divmod(feld, n)
        for nr, lage in kandidaten_fuer(z, s):
            if benutzt[nr]:
                continue
            if symmetrie and nr == 0 and lage != teile[0]:
                continue
            benutzt[nr] = True
            raster[(z, s)] = (nr, lage)
            fertig = rek(feld + 1)
            benutzt[nr] = False
            del raster[(z, s)]
            if fertig:
                return True
        return False

    rek(0)
    return st

In [ ]:
def Z(n):
    return f'{n:,}'.replace(',', ' ')


for datei in ['puzzle_mini1', 'puzzle_mini2']:
    m, n, teile = lies_puzzle('daten/' + datei + '.txt')
    print(f'{m} x {n}:')
    for titel, nachschlagen, symmetrie, alle in [('alle durchprüfen', False, False, True),
                                                 ('nachschlagen', True, False, True),
                                                 ('nachschlagen + Symmetrie', True, True, True)]:
        st = suche_puzzle(m, n, teile, nachschlagen, symmetrie, alle)
        print(f"  {titel:<26} {st['loesungen']} Lösungen, {Z(st['knoten']):>6} Knoten, {Z(st['geprueft']):>7} geprüfte Teile")
        assert ist_loesung(m, n, teile, st['erste'])

2 x 2:
  alle durchprüfen           4 Lösungen,     80 Knoten,   1 216 geprüfte Teile
  nachschlagen               4 Lösungen,     80 Knoten,     698 geprüfte Teile
  nachschlagen + Symmetrie   1 Lösungen,     49 Knoten,     517 geprüfte Teile
3 x 3:
  alle durchprüfen           4 Lösungen,  2 154 Knoten,  77 400 geprüfte Teile
  nachschlagen               4 Lösungen,  2 154 Knoten,  23 486 geprüfte Teile
  nachschlagen + Symmetrie   1 Lösungen,  1 310 Knoten,  16 715 geprüfte Teile


**Lies die Tabelle** (für das 3 × 3-Puzzle): Das Nachschlagen senkt die Zahl der **geprüften Teile** von 77 400 auf 23 486, die Zahl der **Knoten** bleibt gleich (2 154), denn es werden **dieselben** Teile gelegt, nur schneller gefunden. Die Symmetrie senkt die Knoten auf 1 310 und die Lösungen von 4 auf 1 (die vier Drehungen derselben Lösung). Der **Schiedsrichter** `ist_loesung` hat jede gefundene Lösung geprüft.

Für das echte Puzzle mit 4 × 4 oder 5 × 5 Teilen braucht es **mehr**: Bei 4 × 4 sind es schon einige 100 000 Knoten mit dieser einfachen Zeile-für-Zeile-Suche. Wie das Lösungsnotebook zur Aufgabe das verkleinert (zum Beispiel durch eine **bessere Reihenfolge der Felder**, bei der ein Feld früher mehr Nachbarn hat), kannst du dort nachlesen.

#### Frage E2.1

1. Warum bleibt die Zahl der **Knoten** beim Nachschlagen gleich, aber die der **geprüften Teile** sinkt? Welche Größe passt besser zur Laufzeit?
2. Warum darf man Teil 0 auf **eine Lage** festlegen, ohne eine Lösung zu verlieren? Was wäre, wenn Teil 0 eine **Symmetrie** hätte (zum Beispiel `abab`)? Was müsste man ändern?
3. Wie viele Lösungen **bis auf Drehung** hat ein Puzzle mit 4 Drehungen, die alle verschieden sind, wenn die Suche ohne Symmetrie 12 Lösungen findet?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Der Suchbaum ist derselbe: An jedem Knoten werden dieselben Teile ausprobiert. Nur das **Finden** der passenden Teile wird billiger. Für die **Laufzeit** zählt das Produkt aus Knoten und Arbeit pro Knoten, also passt „geprüfte Teile“ besser, wenn man nachschlagen will.
2. Dreht man die ganze Lösung um 90°, wird aus der Lage von Teil 0 eine andere Lage, und alle anderen Teile drehen sich mit. Also gibt es zu **jeder** Lösung eine gleichwertige, in der Teil 0 in der vorgegebenen Lage liegt (im Allgemeinen gibt es genau eine). Hat Teil 0 eine Symmetrie (zum Beispiel `abab` sieht nach 180° gleich aus), hat es **weniger verschiedene Lagen**, und die Lösung kommt nur in 2 Drehungen vor. Dann wäre es falsch, durch 4 zu rechnen, und man müsste sich ein Teil **ohne** Symmetrie für die Festlegung suchen (`lagen(teil)` hat dann 4 Elemente).
3. 12 / 4 = 3 Lösungen bis auf Drehung. Voraussetzung: keine der Lösungen ist in sich symmetrisch (keine Lösung geht durch eine Drehung in sich über). Das ist bei Puzzleteilen, die alle verschieden sind, **immer** so, denn jedes Teil kommt nur einmal vor.

</details>

## Zum Schluss: Sudoku mit MiniZinc

Eine fertige Suche mit geschicktem Verzweigen und Propagation gibt es auch **fertig**: Ein **Solver** wie Gecode oder Chuffed nutzt beides (er wählt Variablen mit kleinem Wertebereich zuerst und schließt aus den Regeln, wie unser `einzelne_setzen`), und du beschreibst **nur das Problem**. Du kennst von den MiniZinc-Arbeitsblättern 2D-Arrays, `forall`, `.dzn`-Dateien, `array2d` und `alldifferent`: Das Sudoku besteht genau aus ihnen. **MiniZinc läuft nicht in JupyterLite:** Kopiere das Modell in die MiniZinc IDE oder in den [MiniZinc Playground](https://play.minizinc.org/) und füge die Daten als `.dzn` dazu (die Python-Zelle darunter erzeugt sie).

Neu ist höchstens die Schreibweise `forall(a, b in {0, 3, 6})`: zwei Laufvariablen zugleich, hier die linke obere Ecke jedes der neun Blöcke, und `[... | i, j in 1..3]` sammelt die neun Felder eines Blocks.


```minizinc
include "alldifferent.mzn";

array[1..9, 1..9] of int: vorgabe;                      % 0 = leer
array[1..9, 1..9] of var 1..9: x;                       % die gesuchte Belegung

% Die Vorgaben stehen fest.
constraint forall(i in 1..9, j in 1..9 where vorgabe[i, j] > 0)(x[i, j] = vorgabe[i, j]);
% In jeder Zeile, jeder Spalte und jedem 3x3-Block stehen lauter verschiedene Ziffern.
constraint forall(i in 1..9)(alldifferent([x[i, j] | j in 1..9]));
constraint forall(j in 1..9)(alldifferent([x[i, j] | i in 1..9]));
constraint forall(a, b in {0, 3, 6})(alldifferent([x[a + i, b + j] | i, j in 1..3]));

solve satisfy;
output [show_int(1, x[i, j]) ++ if j = 9 then "\n" else "" endif | i, j in 1..9];
```

In [ ]:
def als_dzn(raster):
    """Das Sudoku als .dzn-Datei für das MiniZinc-Modell."""
    zahlen = []
    for ziffer in raster:
        zahlen.append(str(ziffer))
    return 'vorgabe = array2d(1..9, 1..9, [' + ', '.join(zahlen) + ']);'


print(als_dzn(lies_sudoku(TEXTE['leicht']))[:75], '...')

vorgabe = array2d(1..9, 1..9, [5, 3, 0, 0, 7, 0, 0, 0, 0, 6, 0, 0, 1, 9, 5, ...


**Die Ausgabe für das „brutale“ Sudoku**, das die Suche mit „erstes freies Feld“ nicht in 50 000 Knoten schafft (Solver Gecode):

**Ausgabe:**

```
987654321
246173985
351928746
128537694
634892157
795461832
519286473
472319568
863745219
```

Der Solver löst alle vier Beispiele, und die Lösungen sind dieselben wie die unserer Python-Suche (das hat `pruefe_sudoku` geprüft und ein Vergleich beim Erzeugen dieses Notebooks bestätigt). Auch die beiden anderen Fragen lassen sich stellen:

- Mit der Option `-a` (alle Lösungen) findet er für das Sudoku **mittel ohne die Vorgabe auf Feld 15** genau **3 Lösungen**, genau wie unser `zaehle_loesungen`.
- Für das unlösbare Sudoku (`leicht` mit einer 1 auf Feld 2) antwortet er `=====UNSATISFIABLE=====`: der **Beweis**, dass es keine Lösung gibt.

**Eigene Suche gegen Solver:** Der Solver ist bei Sudoku **sehr gut**, weil das Problem genau seine Sprache spricht (`alldifferent` erzeugt die Propagation, die wir von Hand geschrieben haben). Er hilft dir **nur**, wenn du die Aufgabe als Variablen und Bedingungen **aufschreiben** kannst. Bei Konfetti oder Komplementär ist das nicht so leicht (eine **Reihenfolge** der Spalten ist schwer zu modellieren, und die Regel „Block zusammenhängend“ braucht Hilfsvariablen), und die eigene Suche mit den Regeln der Aufgabe ist dort die bessere Wahl. Merke: Der Solver ersetzt nicht das **Wissen über die Struktur der Aufgabe**, er rechnet nur schnell.

## Das hast du gelernt

```python
# Eine allgemeine Suche: die Auswahlregel ist ein Argument, die Knoten werden gezählt
def suche_voraus(raster, wahl, zaehler):
    zaehler['knoten'] += 1
    raster = raster[:]                            # eigene Kopie: nichts zurücknehmen
    if not einzelne_setzen(raster):               # erzwungene Felder sofort, Widerspruch = Sackgasse
        return None
    feld = wahl(raster)                           # wo verzweigen? die Regel ist austauschbar
    if feld is None:
        return raster
    for ziffer in kandidaten(raster, feld):
        raster[feld] = ziffer
        ergebnis = suche_voraus(raster, wahl, zaehler)
        if ergebnis is not None:
            return ergebnis
    return None
```

- **Zähle die Knoten** und vergleiche Verfahren mit einer **Tabelle** (Verfahren × Beispiel). Ohne Messung rät man, welche Verbesserung sich lohnt. Prüfe jedes Ergebnis mit dem **Schiedsrichter**. Eine Grenze für die Knoten schützt vor Endlosläufen, ein Abbruch bedeutet **nicht** „keine Lösung“.
- **Wo verzweigen:** Das Feld mit den **wenigsten Möglichkeiten** zuerst (*fail first*). Die Reihenfolge der Entscheidungen bestimmt die **Größe des Suchbaums**: beim Sudoku von mehr als 50 000 auf einige Hundert Knoten, bei den Pentominos auf etwa ein Drittel. Die Auswahlregel ist eine **Funktion als Argument** und lässt sich austauschen.
- **Vorausschauend prüfen:** Erzwungene Schritte sofort ausführen (Propagation) und Widersprüche früh erkennen. Das macht den Knoten teurer, aber den Baum viel kleiner. Bei Konfetti sind es Regeln wie **Regel 1** und die **Blockade**.
- **Teilzustände merken:** Der Zustand ist das, was die Zukunft bestimmt (die Menge der belegten Felder, wie oft jede Art liegt), nicht der Weg dorthin. Sackgassen kommen in eine Menge, Anzahlen in ein Wörterbuch. Gleiche Teile fasst man zusammen. Das wirkt besonders beim **Beweisen von „keine Lösung“**.
- **Symmetrien** früh ausschließen (das X nur links, Teil 0 nur in einer Lage), nicht erst am Ende aussortieren.
- **Eine Lösung, alle Lösungen, keine Lösung:** „keine“ beweist nur die **vollständige** Suche; für die Eindeutigkeit reicht es, bis 2 zu zählen.
- Ein **Solver** (MiniZinc) nimmt dir Auswahlregel und Propagation ab, wenn du die Aufgabe als Variablen und Bedingungen aufschreiben kannst.

### Aufgaben der 2. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Konfetti](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/43_Konfetti/konfetti_loesung.ipynb) (43. BwInf): Spalten von links nach rechts legen, Sackgassen merken, die Regeln im Original (mit unleserlichen Einträgen und Annas Zeile)
- [Komplementär](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/31_Komplementaer/komplementaer_loesung.ipynb) (31.): Puzzle, Nachschlagen statt Durchprobieren und Symmetrie
- [Stromrallye](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/38_Stromrallye/stromrallye_loesung.ipynb) (38.): ein Spiel mit Batterien, Sackgassen merken
- [Die Kunst der Fuge](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/36_DieKunstDerFuge/fuge_loesung.ipynb) (36.): Backtracking mit gutem Abschneiden
- [Alles Käse](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/41_AllesKaese/kaese_loesung.ipynb) (41.): ein weiteres Beispiel für vollständige Suche

Weitere Aufgaben stehen in der Werkzeugübersicht der 2. Runde (Abschnitt „Vollständige Suche bei großen Fällen“). Das nächste Kern-Notebook ist **07 „Große Zustandsräume“**: Dort geht es um **Breitensuche, A\*, die Suche von beiden Seiten** und die **Tiefenschranke**, wenn der Zustandsraum zu groß ist, um ihn ganz zu durchsuchen.